In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        return x[:, :, :-self.chomp_size]

class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super().__init__()

        padding = (kernel_size - 1) * dilation

        self.net = nn.Sequential(
            nn.Conv1d(
                in_channels,
                out_channels,
                kernel_size,
                padding=padding,
                dilation=dilation
            ),
            Chomp1d(padding),
            nn.ReLU(),
            nn.Conv1d(
                out_channels,
                out_channels,
                kernel_size,
                padding=padding,
                dilation=dilation
            ),
            Chomp1d(padding),
            nn.ReLU()
        )

        self.downsample = None
        if in_channels != out_channels:
            self.downsample = nn.Conv1d(in_channels, out_channels, 1)

    def forward(self, x):
        out = self.net(x)
        residual = x if self.downsample is None else self.downsample(x)
        return out + residual

class TCN(nn.Module):
    def __init__(self, input_channels, num_channels, kernel_size, num_classes):
        super().__init__()

        layers = []

        for i in range(len(num_channels)):
            dilation = 2 ** i
            in_channels = input_channels if i == 0 else num_channels[i - 1]
            out_channels = num_channels[i]

            layers.append(
                TemporalBlock(
                    in_channels,
                    out_channels,
                    kernel_size,
                    dilation
                )
            )

        self.network = nn.Sequential(*layers)
        self.fc = nn.Linear(num_channels[-1], num_classes)

    def forward(self, x):
        x = self.network(x)
        x = x[:, :, -1]
        x = self.fc(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TCN(
    input_channels=1,
    num_channels=[32, 64, 64],
    kernel_size=3,
    num_classes=2
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 5

for epoch in range(epochs):
    inputs = torch.randn(64, 1, 100).to(device)
    labels = torch.randint(0, 2, (64,)).to(device)

    optimizer.zero_grad()

    outputs = model(inputs)

    loss = criterion(outputs, labels)

    loss.backward()

    optimizer.step()

    print("Epoch:", epoch + 1, "Loss:", loss.item())

model.eval()

with torch.no_grad():
    test_input = torch.randn(10, 1, 100).to(device)
    predictions = model(test_input)
    predicted = torch.argmax(predictions, dim=1)

print(predicted)

ModuleNotFoundError: No module named 'torch'